<a href="https://colab.research.google.com/github/bermanlabemory/unsupervised-behavior-tutorials/blob/main/00_colab_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0.&nbsp; Colab check 

**Time: ~2 minutes.** This notebook just makes sure your Google Colab runtime can do everything
we'll for the workshop. Run every cell top-to-bottom (Shift+Enter, or
`Runtime → Run all`). If the last cell prints a happy message, you're ready. If something
breaks, send us the error before the session and we'll sort it out.

## 0.1&nbsp; What machine did Google give us?

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0], "on", platform.system())

# A GPU is optional for this workshop, but nice to have (it speeds up UMAP + autoencoders).
# To request one: Runtime -> Change runtime type -> Hardware accelerator -> GPU, then rerun.
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "none (that's fine!)")

## 0.2&nbsp; Install the few packages Colab doesn't ship

In [ ]:
# Quietly install what's missing. This takes ~30-60 s.
!pip install -q hdf5storage easydict umap-learn 2>/dev/null
print("packages installed")

## 0.3&nbsp; Grab the analysis engine

We'll use **motionmapperpy** &mdash; a GPU-friendly Python implementation of MotionMapper.
We download it by cloning its GitHub repository (this also brings along the small example
datasets we'll use).

In [ ]:
import os, sys, types
if not os.path.exists("motionmapperpy"):
    !git clone -q https://github.com/bermanlabemory/motionmapperpy
!pip install -q hdf5storage easydict umap-learn

# This check doesn't need video. The released package imports moviepy at load time, and Colab's
# moviepy/ffmpeg stack tries to download an ancient ffmpeg from a dead URL -- so we stub moviepy
# out. (The notebooks that actually render videos install a working moviepy instead.)
def _stub(name, **attrs):
    m = sys.modules.get(name) or types.ModuleType(name)
    for k, v in attrs.items():
        setattr(m, k, v)
    sys.modules[name] = m
    return m
_stub("moviepy"); _stub("moviepy.editor", VideoClip=object, VideoFileClip=object)
_stub("moviepy.video"); _stub("moviepy.video.io")
_stub("moviepy.video.io.bindings", mplfig_to_npimage=lambda *a, **k: None)

# Import straight from the clone -- avoids the "setup.py install" empty-namespace trap; no restart.
sys.path.insert(0, os.path.abspath("motionmapperpy"))
for _m in [k for k in list(sys.modules) if k.startswith("motionmapperpy")]:
    del sys.modules[_m]
print("motionmapperpy ready")

> **No restart needed.** The setup cell imports motionmapperpy straight from the cloned folder. If
> `import motionmapperpy` ever fails, just re-run the setup cell above — do **not** use *Restart
> session*, which would undo the `sys.path` line. Your files are kept either way.

## 0.4&nbsp; Set-up Test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import motionmapperpy as mmpy

# Make a tiny fake postural time series and run one real step of the pipeline (a wavelet
# transform). If this returns an array, the whole engine works.
x = np.cumsum(np.random.randn(2000, 3), axis=0)      # 3 "postural modes" over 2000 frames
params = mmpy.setRunParameters()
wavelets, freqs = mmpy.findWavelets(x, 3, params.omega0, 25, 100, 50, 1,
                                    params.numProcessors, -1)

fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(wavelets[:500].T, aspect="auto", cmap="PuRd", origin="lower")
ax.set_title("a wavelet spectrogram of fake data — if you can see this, you're ready")
ax.set_xlabel("frame"); ax.set_ylabel("wavelet channel")
plt.show()

print("\n✅ All set!")